# Practice Lab: Designing for Reliability (Lesson 12.2)

Module 12 . 4 exercises . Retry + breaker + cache + chaos testing


# Lesson 12.2: Designing for Reliability

4 exercises: 2E + 1M + 1C

All exercises run **locally** (pure Python simulations).


In [ ]:
import random, time, hashlib
random.seed(42)


---
## Exercise 1 (Easy): Retry Statistics


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class RetryTracker:
    def __init__(self,retries=3,fail=0.40):
        self.retries=retries;self.fail=fail;self.stats={f"a{i}":{"ok":0,"fail":0} for i in range(retries+1)}
        self.total=0;self.success=0;self.retry_count=0
    def call(self):
        self.total+=1
        for a in range(self.retries+1):
            if random.random()>self.fail:
                self.stats[f"a{a}"]["ok"]+=1;self.success+=1
                if a>0: self.retry_count+=a
                return a
            self.stats[f"a{a}"]["fail"]+=1
            if a<self.retries: self.retry_count+=1
        return -1

t=RetryTracker(3,0.40)
print("Retry Statistics (100 requests, 40% fail):")
for _ in range(100): t.call()
cum=0
for i in range(4):
    s=t.stats[f"a{i}"]["ok"]; cum+=s
    label="1st try" if i==0 else f"retry {i}"
    print(f"  {label:<10} ok={s:>3} ({s:.0f}%) cumulative={cum:.0f}%")
print(f"\n  Success: {t.success}/100 | Retries: {t.retry_count} | Avg: {t.retry_count/100:.2f}/req")
print(f"  Backoff: 0s -> 0.1s -> 0.2s -> 0.4s + jitter")


</details>


---
## Exercise 2 (Easy): Breaker Dashboard


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import time

class Breaker:
    def __init__(self,th=3,reset=0.3):
        self.state="CLOSED";self.th=th;self.reset=reset;self.fails=0;self.last_fail=0;self.blocked=0
    def call(self,ok):
        if self.state=="OPEN":
            if time.time()-self.last_fail>=self.reset: self.state="HALF-OPEN"
            else: self.blocked+=1; return self.state,"BLOCKED"
        if ok:
            if self.state=="HALF-OPEN": self.state="CLOSED"; self.fails=0
            return self.state,"OK"
        self.fails+=1; self.last_fail=time.time()
        if self.fails>=self.th: self.state="OPEN"
        return self.state,"FAIL"

b=Breaker(3,0.3)
calls=[True]*4+[False]*6+[True]*10
print("Circuit Breaker Dashboard:")
print(f"  {'Call':>4} {'Input':>6} {'State':>10} {'Result':>8}")
for i,ok in enumerate(calls):
    if i==10: time.sleep(0.35)
    state,res=b.call(ok)
    print(f"  {i+1:>4} {'OK' if ok else 'FAIL':>6} {state:>10} {res:>8}")
print(f"\n  Blocked: {b.blocked} calls | Saved: ~{b.blocked*10}s of timeouts")


</details>


---
## Exercise 3 (Medium): Cache with TTL


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import time, hashlib

class Cache:
    def __init__(self,ttl=3600): self.store={}; self.ttl=ttl; self.stats={"hit":0,"miss":0,"exp":0}
    def _k(self,p): return hashlib.md5(p.lower().strip().encode()).hexdigest()[:16]
    def get(self,p):
        k=self._k(p)
        if k in self.store:
            if time.time()<self.store[k]["exp"]: self.stats["hit"]+=1; return self.store[k]["r"]
            self.stats["exp"]+=1; del self.store[k]
        self.stats["miss"]+=1; return None
    def put(self,p,r,ttl=None): self.store[self._k(p)]={"r":r,"exp":time.time()+(ttl or self.ttl)}

c=Cache(3600)
for p,r in [("Course price?","14999"),("Prerequisites?","Python"),("How long?","146hrs"),("Refund?","7d full"),("Classes?","Jul 2026")]:
    c.put(p,r)
c.put("Flash sale?","9999",ttl=0.01); time.sleep(0.02)

print("Cache with TTL:")
for q in ["Course price?","Tell me prereqs","Prerequisites?","How long?","Compare courses","Refund?",
    "Flash sale?","Course price?","Design RAG","Classes?","Refund?","Cost?","Prerequisites?",
    "Explain transformers","How long?","Course price?","Deploy?","Classes?","Refund?","Certificate?"]:
    r=c.get(q); status="HIT" if r else "MISS"
    if r is None: c.put(q,f"Resp:{q[:15]}")
    print(f"  {status:>4}: {q[:30]}")

total=sum(c.stats.values()); hr=c.stats["hit"]/max(total,1)
print(f"\n  Stats: {c.stats} | Hit rate: {hr:.0%}")
print(f"  Saved: ${c.stats['hit']*0.003:.4f} of ${total*0.003:.4f}")


</details>


---
## Exercise 4 (Challenge): Chaos Testing


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class ChaosClient:
    def __init__(self,providers):
        self.p=providers; self.stats={p["n"]:{"ok":0,"fail":0,"blk":0} for p in providers}
        self.stats["degraded"]=0; self.breaker_f={p["n"]:0 for p in providers}
        self.breaker_open={p["n"]:False for p in providers}; self.total=0; self.served=0
    def _try(self,prov):
        n=prov["n"]
        if self.breaker_open[n]:
            self.breaker_f[n]+=1
            if self.breaker_f[n]>10: self.breaker_open[n]=False; self.breaker_f[n]=0
            self.stats[n]["blk"]+=1; return None
        for _ in range(3):
            if random.random()>prov["fr"]: self.stats[n]["ok"]+=1; return n
            self.stats[n]["fail"]+=1
        self.breaker_open[n]=True; return None
    def call(self):
        self.total+=1
        for prov in self.p:
            r=self._try(prov)
            if r: self.served+=1; return {"provider":r,"degraded":False}
        self.stats["degraded"]+=1; self.served+=1
        return {"provider":"degraded","degraded":True}

cl=ChaosClient([{"n":"gemini","fr":0.50},{"n":"openai","fr":0.30},{"n":"claude","fr":0.10}])
print("Chaos Testing (1000 requests):")
deg=0
for i in range(1000):
    r=cl.call()
    if r["degraded"]: deg+=1

print(f"  Served: {cl.served}/1000 ({cl.served/10:.1f}%)")
print(f"  Degraded: {deg} ({deg/10:.1f}%)")
for n in ["gemini","openai","claude"]:
    s=cl.stats[n]; print(f"  {n:<10} ok={s['ok']:>4} fail={s['fail']:>4} blocked={s['blk']:>4}")
sla=cl.served/cl.total*100
print(f"\n  SLA: {sla:.1f}% (target: 99.9%) -> {'MEETS' if sla>=99.9 else 'CHECK'}")
print(f"  3-provider fallback + degradation = near-100%")


</details>
